In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.iteration import iterate_calibration, theoretical_error_decay
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensors import NOISE_FREE, SensorConfig
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base, truth = ve_table(params=BASE_MAP_SURFACE), ve_table(params=TRUE_SURFACE)

gains = (0.2, 0.4, 0.6, 0.8, 1.0)
runs = {}

In [ ]:
"""
Sweep the damping gain of the fixed-point correction.

The update is VE_new = VE_old * (1 + k * (AFR_measured / AFR_target - 1)).
A gain of one applies the full indicated correction; smaller values approach
the solution more cautiously.
"""

print(f"{'gain':>5} {'passes':>7} {'conv':>6} {'RMSE':>7} {'visited':>8} {'AFR%':>6}")
for gain in gains:
    history = iterate_calibration(
        train, base, gain=gain, max_passes=12, method="bilinear", tolerance=1e-4
    )
    runs[gain] = history
    t = evaluate_table(history.final_table, truth)
    a = evaluate_afr(history.final_table, test)
    print(f"{gain:5.1f} {history.n_passes:7d} {str(history.converged):>6} "
          f"{t.rmse:7.2f} {t.rmse_visited:8.2f} {a.mean_abs_pct:6.2f}")

In [ ]:
"""
Plot table error and per-pass update size against iteration.
"""

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for gain, history in runs.items():
    rmse = [evaluate_table(t, truth).rmse for t in history.tables]
    axes[0].plot(rmse, marker="o", ms=4, label=f"k = {gain}")

    axes[1].semilogy(
        range(1, len(history.max_change) + 1), history.max_change,
        marker="o", ms=4, label=f"k = {gain}",
    )

axes[0].set_xlabel("pass")
axes[0].set_ylabel("table RMSE (VE points)")
axes[0].set_title("Accuracy against iteration")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(1e-4, color="black", ls="--", lw=1, label="tolerance")
axes[1].set_xlabel("pass")
axes[1].set_ylabel("largest cell change")
axes[1].set_title("Convergence of the update")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()

In [ ]:
"""
Compare the observed decay rate against the analytic fixed-point rate.

For a linear fixed-point iteration with damping gain k, error after n passes
falls as (1 - k)**n. Deviation from that curve measures how far the real
problem departs from the idealised one.
"""

gain = 0.5
ideal = iterate_calibration(
    train, base, gain=gain, max_passes=8, method="bilinear",
    sensor_config=NOISE_FREE, tolerance=1e-12,
)
noisy = iterate_calibration(
    train, base, gain=gain, max_passes=8, method="bilinear",
    sensor_config=SensorConfig(), tolerance=1e-12,
)

predicted = theoretical_error_decay(gain, len(ideal.max_change))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(predicted[1:] * ideal.rms_change[0] / predicted[1],
            "k--", marker="s", ms=4, label="theory: (1-k)^n")
ax.semilogy(range(0, len(ideal.rms_change) ), ideal.rms_change,
            marker="o", ms=4, label="noise free")
ax.semilogy(range(0, len(noisy.rms_change) ), noisy.rms_change,
            marker="o", ms=4, label="with sensor noise")

ax.set_xlabel("pass")
ax.set_ylabel("RMS change per pass")
ax.set_title(f"Convergence against theoretical rate, k = {gain}")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
"""
Run the iteration far past its useful point to characterise the limit
"""

from vetuner.engine_model import simulate_engine
from vetuner.sensors import NOISE_FREE, SensorConfig, apply_sensor_model

sim = simulate_engine(train, base)
clean = apply_sensor_model(sim, config=NOISE_FREE, seed=0)
noisy = apply_sensor_model(sim, config=SensorConfig(), seed=0)

print(f"AFR std, clean: {clean.afr.std():.4f}")
print(f"AFR std, noisy: {noisy.afr.std():.4f}")
print(f"max difference: {abs(clean.afr - noisy.afr).max():.4f}")

In [ ]:

import numpy as np

diff = np.abs(clean.afr - noisy.afr)
worst = int(np.argmax(diff))

print(f"worst at t = {clean.time_s[worst]:.2f} s, difference {diff[worst]:.2f}")
print(f"clean {clean.afr[worst]:.2f}, noisy {noisy.afr[worst]:.2f}")
print(f"rpm {sim.rpm[worst]:.0f}, tps {sim.tps[worst]:.3f}, fuel cut {sim.fuel_cut[worst]}")
print(f"\nmedian difference: {np.median(diff):.4f}")
print(f"95th percentile:   {np.percentile(diff, 95):.4f}")
print(f"difference where not in fuel cut: {diff[~sim.fuel_cut].max():.4f}")

In [ ]:
from vetuner.iteration import iterate_calibration
from vetuner.sensors import NOISE_FREE

for method in ("per_cell", "bilinear"):
    h = iterate_calibration(
        train, base, gain=0.8, max_passes=25, method=method,
        sensor_config=NOISE_FREE, tolerance=1e-4,
    )
    tail = h.max_change[-6:]
    print(f"{method:10s} passes {h.n_passes:3d}  converged {h.converged}")
    print(f"           last six changes: {['%.2e' % c for c in tail]}")

In [ ]:
import numpy as np

from vetuner.iteration import iterate_calibration
from vetuner.metrics import evaluate_table
from vetuner.sensors import NOISE_FREE

h = iterate_calibration(
    train, base, gain=0.8, max_passes=25, method="per_cell",
    sensor_config=NOISE_FREE, tolerance=1e-12,
)

rmse = [evaluate_table(t, truth).rmse for t in h.tables]
print("RMSE by pass:", [f"{r:.3f}" for r in rmse[::5]])

visited = h.tables[-1] != h.tables[0]
drift = h.tables[-1][visited] - h.tables[5][visited]
print(f"\nmean drift, pass 5 to 25: {drift.mean():+.5f}")
print(f"fraction drifting the same way: {max((drift > 0).mean(), (drift < 0).mean()):.2f}")
print(f"mean VE in visited cells: {h.tables[-1][visited].mean():.4f}")